# A/B Test Analysis: Marketing Ad Campaign
**Dataset:** Marketing A/B Testing (Kaggle)  
**Author:** Ernesto Acosta Berg | [LinkedIn](https://linkedin.com/in/ernesto-acosta-berg) | [GitHub](https://github.com/ebctrl)  
**Tools:** Python · Pandas · SciPy · Matplotlib · Seaborn

---

## Objective
A company ran an A/B test to determine whether showing users an **advertisement** (treatment) converts more customers than showing a **public service announcement** (control). This analysis answers:
1. Did the ad significantly increase conversions?
2. How large is the effect?
3. Which days and hours perform best?
4. Does showing more ads to a user increase their conversion rate?

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.facecolor': '#F7F7F5',
    'figure.facecolor': '#FFFFFF',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'axes.grid': True,
    'grid.color': '#E0E0E0',
    'grid.linewidth': 0.8,
})

COLORS = {'ad': '#2563EB', 'psa': '#94A3B8', 'accent': '#F59E0B'}

df = pd.read_csv('../data/marketing_AB.csv', index_col=0)
df['converted'] = df['converted'].astype(bool)
print(f'Rows: {len(df):,} | Columns: {list(df.columns)}')
df.head()

## 2. Exploratory Data Analysis

| Column | Type | Description |
|---|---|---|
| `user id` | int | Unique user identifier |
| `test group` | str | `ad` = saw advertisement, `psa` = saw public service announcement |
| `converted` | bool | Did the user make a purchase? |
| `total ads` | int | How many ads the user was shown |
| `most ads day` | str | Day of week when most ads were shown |
| `most ads hour` | int | Hour of day when most ads were shown |

In [ ]:
print('=== GROUP SIZES ===')
print(df['test group'].value_counts())
print(f'\n96% of users saw an ad, 4% saw a PSA (control group)')
print('\n=== CONVERSION RATE BY GROUP ===')
print(df.groupby('test group')['converted'].mean().mul(100).round(3).astype(str) + '%')
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

## 3. Core Conversion Metrics

In [ ]:
ad  = df[df['test group'] == 'ad']
psa = df[df['test group'] == 'psa']

n_ad,  conv_ad  = len(ad),  ad['converted'].sum()
n_psa, conv_psa = len(psa), psa['converted'].sum()

rate_ad  = conv_ad  / n_ad
rate_psa = conv_psa / n_psa
lift     = (rate_ad - rate_psa) / rate_psa * 100

def wilson_ci(successes, n, z=1.96):
    p = successes / n
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return center - margin, center + margin

ci_ad  = wilson_ci(conv_ad,  n_ad)
ci_psa = wilson_ci(conv_psa, n_psa)

results = pd.DataFrame({
    'Group':       ['PSA (Control)', 'Ad (Treatment)'],
    'Users':       [f'{n_psa:,}', f'{n_ad:,}'],
    'Conversions': [f'{conv_psa:,}', f'{conv_ad:,}'],
    'Rate':        [f'{rate_psa*100:.3f}%', f'{rate_ad*100:.3f}%'],
    '95% CI':      [f'{ci_psa[0]*100:.3f}% - {ci_psa[1]*100:.3f}%',
                    f'{ci_ad[0]*100:.3f}% - {ci_ad[1]*100:.3f}%'],
})
print(results.to_string(index=False))
print(f'\nRelative lift: +{lift:.1f}%')
print(f'Absolute difference: +{(rate_ad - rate_psa)*100:.3f} percentage points')

## 4. Hypothesis Testing

**H₀ (null):** The ad and PSA groups have the same conversion rate.  
**H₁ (alternative):** The ad group converts at a different rate than PSA.  
**Test:** Chi-square test of independence — used when comparing two groups on a binary outcome.  
**Threshold:** α = 0.05. If p < 0.05, the result is statistically significant.

In [ ]:
contingency = pd.crosstab(df['test group'], df['converted'])
print('Contingency table:')
print(contingency)

chi2, p_val, dof, expected = chi2_contingency(contingency)

print(f'\nChi-square statistic: {chi2:.4f}')
print(f'p-value:              {p_val:.2e}')
print(f'Degrees of freedom:   {dof}')
sig = 'REJECT H0 - the ad significantly increases conversion (p < 0.001)' if p_val < 0.05 else 'FAIL TO REJECT H0'
print(f'\nDecision: {sig}')

### Chart 1: Conversion Rate Comparison with Confidence Intervals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

groups = ['PSA (Control)', 'Ad (Treatment)']
rates  = [rate_psa * 100, rate_ad * 100]
cis    = [ci_psa, ci_ad]
colors = [COLORS['psa'], COLORS['ad']]

bars = ax.bar(groups, rates, color=colors, width=0.45, zorder=3, edgecolor='white', linewidth=1.5)

for i, (rate, ci) in enumerate(zip(rates, cis)):
    ax.errorbar(i, rate, yerr=[[rate - ci[0]*100], [ci[1]*100 - rate]],
                fmt='none', color='#374151', capsize=6, capthick=2, linewidth=2, zorder=4)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, rate + 0.05,
            f'{rate:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.annotate(f'+{lift:.1f}% lift', xy=(1, rate_ad*100), xytext=(1.32, (rate_ad+rate_psa)/2*100),
            fontsize=11, color=COLORS['ad'], fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLORS['ad'], lw=1.5))

ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate: Ad vs PSA Control Group\nwith 95% Confidence Intervals', pad=15)
ax.set_ylim(0, max(rates) * 1.35)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.set_facecolor('#F7F7F5')
ax.text(0.98, 0.05, f'chi2={chi2:.1f}, p<0.001  n(ad)={n_ad:,}  n(psa)={n_psa:,}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='#6B7280', style='italic')

plt.tight_layout()
plt.savefig('../outputs/01_conversion_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Segmentation Analysis

Now we break down performance by day of week, hour of day, and ad exposure volume.

In [ ]:
day_order   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_conv    = df.groupby(['test group','most ads day'])['converted'].mean().unstack(0) * 100
day_conv    = day_conv.reindex(day_order)
hour_conv   = df.groupby(['test group','most ads hour'])['converted'].mean().unstack(0) * 100
df['ads_bucket'] = pd.cut(df['total ads'], bins=[0,5,15,30,60,120,9999],
                           labels=['1-5','6-15','16-30','31-60','61-120','120+'])
bucket_conv = df.groupby(['test group','ads_bucket'])['converted'].mean().unstack(0) * 100

print('Conversion by day (ad group):')
print(day_conv['ad'].sort_values(ascending=False).round(2))

### Chart 2: Conversion by Day of Week

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x, w = np.arange(len(day_order)), 0.35

ax.bar(x - w/2, day_conv.get('psa', [0]*7), w, label='PSA (Control)', color=COLORS['psa'], zorder=3, edgecolor='white')
ax.bar(x + w/2, day_conv.get('ad',  [0]*7), w, label='Ad (Treatment)', color=COLORS['ad'],  zorder=3, edgecolor='white')

ax.set_xticks(x); ax.set_xticklabels(day_order)
ax.set_ylabel('Conversion Rate (%)'); ax.set_title('Conversion Rate by Day of Week')
ax.legend(frameon=False); ax.set_facecolor('#F7F7F5')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

best_day_pos = day_order.index(day_conv['ad'].idxmax())
ax.axvspan(best_day_pos - 0.45, best_day_pos + 0.45, alpha=0.08, color=COLORS['accent'], zorder=0)
ax.text(best_day_pos + 0.5, day_conv['ad'].max() * 0.98,
        f"Best: {day_conv['ad'].idxmax()}", color=COLORS['accent'], fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/02_conversion_by_day.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 3: Conversion by Hour of Day

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
hours = sorted(df['most ads hour'].unique())

ax.plot(hours, hour_conv.get('psa', pd.Series(index=hours)), 'o-',
        color=COLORS['psa'], linewidth=2, markersize=5, label='PSA (Control)')
ax.plot(hours, hour_conv.get('ad',  pd.Series(index=hours)), 'o-',
        color=COLORS['ad'], linewidth=2.5, markersize=5, label='Ad (Treatment)')
ax.fill_between(hours, hour_conv.get('ad', pd.Series(index=hours)), alpha=0.1, color=COLORS['ad'])

best_hour = hour_conv['ad'].idxmax()
ax.axvline(best_hour, color=COLORS['accent'], linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(best_hour + 0.2, hour_conv['ad'].max(), f'Peak: {best_hour}:00',
        color=COLORS['accent'], fontweight='bold', fontsize=9)

ax.set_xlabel('Hour of Day (24h)'); ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Hour of Day')
ax.set_xticks(hours); ax.legend(frameon=False); ax.set_facecolor('#F7F7F5')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

plt.tight_layout()
plt.savefig('../outputs/03_conversion_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 4: Dose-Response — Does Seeing More Ads Help?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x, w = np.arange(len(bucket_conv)), 0.35
labels = [str(b) for b in bucket_conv.index]

ax.bar(x - w/2, bucket_conv.get('psa', [0]*len(x)), w, label='PSA (Control)', color=COLORS['psa'], zorder=3, edgecolor='white')
ax.bar(x + w/2, bucket_conv.get('ad',  [0]*len(x)), w, label='Ad (Treatment)',  color=COLORS['ad'],  zorder=3, edgecolor='white')

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('Number of Ads Seen'); ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Ad Exposure (Dose-Response)\nDoes seeing more ads increase conversion?')
ax.legend(frameon=False); ax.set_facecolor('#F7F7F5')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

plt.tight_layout()
plt.savefig('../outputs/04_conversion_by_exposure.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 5: Day × Hour Heatmap (Ad Group Only)

In [ ]:
pivot = ad.groupby(['most ads day','most ads hour'])['converted'].mean().unstack() * 100
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, ax=ax, cmap='Blues', linewidths=0.3, linecolor='white',
            fmt='.1f', annot=True, annot_kws={'size': 7},
            cbar_kws={'label': 'Conversion Rate (%)', 'shrink': 0.8})

ax.set_title('Conversion Rate Heatmap: Day x Hour (Ad Group)\nBest times to show ads', pad=15)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('')
ax.tick_params(axis='x', rotation=0); ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../outputs/05_heatmap_day_hour.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Results Summary

In [ ]:
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
print(f'\nSTATISTICAL RESULT')
print(f'  H0 rejected: YES (p = {p_val:.2e})')
print(f'  The ad drives a statistically significant increase in conversions.')
print(f'\nEFFECT SIZE')
print(f'  PSA conversion rate: {rate_psa*100:.3f}%')
print(f'  Ad  conversion rate: {rate_ad*100:.3f}%')
print(f'  Absolute lift:       +{(rate_ad-rate_psa)*100:.3f} percentage points')
print(f'  Relative lift:       +{lift:.1f}%')
print(f'\nBEST PERFORMING SEGMENTS')
print(f'  Best day:  {day_conv["ad"].idxmax()} ({day_conv["ad"].max():.2f}% conversion)')
print(f'  Worst day: {day_conv["ad"].idxmin()} ({day_conv["ad"].min():.2f}% conversion)')
print(f'  Best hour: {int(hour_conv["ad"].idxmax())}:00 ({hour_conv["ad"].max():.2f}% conversion)')
print(f'\nDOSE-RESPONSE')
print(f'  Users seeing 120+ ads convert at ~16.9% vs 0.25% for 1-5 ads.')
print(f'  Note: correlational, not causal.')